In [ ]:
from pathlib import Path
import pandas as pd
import chromadb
from aatm.data_models import ExpressionMetadata
import dotenv
import os
from google import genai
from chromadb import Documents, EmbeddingFunction, Embeddings
from tqdm import tqdm

dotenv.load_dotenv()

class GoogleEmbeddingFunction(EmbeddingFunction):
    def __init__(self, model: str, *args, **kwargs):
        self.client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
        self.model = model

    def __call__(self, input: Documents) -> Embeddings:
        response = self.client.models.embed_content(
            model=self.model,
            contents=input,
        )
        return [emb.values for emb in response.embeddings]    

client = chromadb.PersistentClient()
try: 
    client.delete_collection("expressions")
except:
    pass

collection = client.get_or_create_collection(
    "expressions", 
    embedding_function=GoogleEmbeddingFunction(
            model="gemini-embedding-001"
        )
    )

batch_size = 100 # max batch size = 100
datasets_base_path = Path("datasets")
for dataset_path in datasets_base_path.glob("*.csv"):
    expression_origin = dataset_path.stem
    df = pd.read_csv(dataset_path)
    for i in tqdm(
        range(0, len(df), batch_size), 
        desc=f"Adding embeddings for {expression_origin}", 
        ):
        records = df.iloc[i : i + batch_size].to_dict("records")
        records = [ExpressionMetadata(**record, expression_origin=expression_origin) for record in records]
        collection.add(
            ids=[record.expression_id for record in records],
            documents=[record.expression for record in records],
            metadatas=[record.to_dict() for record in records],
        )

Adding embeddings for mapped_non_standard_concept:   1%|          | 3/416 [00:07<18:09,  2.64s/it]


KeyboardInterrupt: 

In [17]:
collection.get(include=['embeddings'])['embeddings'].shape

(10, 3072)

In [ ]:
from aatm.data_models import ExpressionMetadata, ExpressionOrigin, StandardVocabulary


model = ExpressionMetadata(
    expression="hello",
    expression_concept_id="world",
    expression_origin=ExpressionOrigin.STANDARD_CONCEPT,
    std_concept_id="hello",
    std_concept_name="world",
    std_vocabulary_id=StandardVocabulary.SNOMED,
    std_vocabulary_code="hello",
).to_dict()

model

{'expression_id': '090d2d0cf8c87fb4',
 'expression': 'hello',
 'expression_concept_id': 'world',
 'expression_origin': 'standard_concept',
 'std_concept_id': 'hello',
 'std_concept_name': 'world',
 'std_vocabulary_id': 'SNOMED',
 'std_vocabulary_code': 'hello'}

: 

In [ ]:
ExpressionMetadata(**model)

ExpressionMetadata(expression_id='090d2d0cf8c87fb4', expression='hello', expression_concept_id='world', expression_origin=<ExpressionOrigin.STANDARD_CONCEPT: 'standard_concept'>, std_concept_id='hello', std_concept_name='world', std_vocabulary_id=<StandardVocabulary.SNOMED: 'SNOMED'>, std_vocabulary_code='hello')

: 